In [ ]:
 
from src.schema.models import (
    SQLTableSchema, MongoCollectionSchema, CrossSourceRelation
)
from config.settings import settings
from openai import OpenAI
from dotenv import load_dotenv
import json
import re
import logging

load_dotenv()
client = OpenAI(
            base_url="https://router.huggingface.co/v1",
            api_key=settings.HF_TOKEN,
        )


response = client.chat.completions.create(
            model=settings.HF_MODEL,
            temperature=0,
            # NOTE: response_format is intentionally NOT set to json_object
            # because json_object mode forces a dict wrapper and breaks plain arrays
            messages=[
                {
                    "role": "system",
                    "content": "you are professinal database enginner",
                },
                {
                    "role": "user",
                    "content": (
                      "wrote queary to get the highest number of orders"
                    ),
                },
            ],
        )

response

AttributeError: 'ChatCompletion' object has no attribute 'content'

In [ ]:
ChatCompletion(id='chatcmpl-2aea14e8-b1e6-4754-a0d5-00cccd801f69', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='To get the highest number of orders, you can use the following SQL query:\n\n```sql\nSELECT MAX(order_count) AS highest_order_count\nFROM (\n  SELECT COUNT(*) AS order_count\n  FROM orders\n  GROUP BY customer_id\n) AS subquery;\n```\n\nHowever, this query will return the highest number of orders for each customer. If you want to get the highest number of orders overall, you can use the following query:\n\n```sql\nSELECT MAX(order_count) AS highest_order_count\nFROM (\n  SELECT COUNT(*) AS order_count\n  FROM orders\n  GROUP BY customer_id\n) AS subquery;\n```\n\nBut if you want to get the customer id and the highest number of orders, you can use the following query:\n\n```sql\nSELECT customer_id, order_count\nFROM (\n  SELECT customer_id, COUNT(*) AS order_count,\n         RANK() OVER (ORDER BY COUNT(*) DESC) AS rank\n  FROM orders\n  GROUP BY customer_id\n) AS subquery\nWHERE rank = 1;\n```\n\nThis query uses the RANK() function to rank the customers by the number of orders in descending order. The WHERE clause then selects the customer with the highest number of orders (rank = 1).\n\nAlternatively, you can use the DENSE_RANK() function to get the same result:\n\n```sql\nSELECT customer_id, order_count\nFROM (\n  SELECT customer_id, COUNT(*) AS order_count,\n         DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS rank\n  FROM orders\n  GROUP BY customer_id\n) AS subquery\nWHERE rank = 1;\n```\n\nThis query is similar to the previous one, but it uses the DENSE_RANK() function instead of RANK(). The DENSE_RANK() function assigns the same rank to customers with the same number of orders, whereas the RANK() function assigns consecutive ranks.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774206249, model='llama3.1-8b', object='chat.completion', service_tier=None, system_fingerprint='fp_5198798116a66ebf301b', usage=CompletionUsage(completion_tokens=386, prompt_tokens=53, total_tokens=439, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=None, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cached_tokens=0)), time_info={'queue_time': 0.063616098, 'prompt_time': 0.00406719, 'completion_time': 0.154368131, 'total_time': 0.22324156761169434, 'created': 1774206249.5106754})